In [1]:
import tensorflow as tf
from tensorflow.keras.models import model_from_json

# load model
json = 'model_rnn_GRU.120_Dense.5010_LSTMKernelInit.VarianceScaling_DenseKernelInit.lecun_uniformKRl1.0_KRl2.0_recAct.sigmoid_arch.json'
with open(json, 'r') as json_file:
    load = json_file.read()
model = model_from_json(load)

# load weights
weights = 'model_rnn_GRU.120_Dense.5010_LSTMKernelInit.VarianceScaling_DenseKernelInit.lecun_uniformKRl1.0_KRl2.0_recAct.sigmoid_weights.h5'
model.load_weights(weights)

model.summary()

2026-04-23 15:43:56.447383: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-23 15:43:56.449659: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-23 15:43:56.489392: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-23 15:43:56.489414: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-23 15:43:56.490494: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 15, 6)]           0         
                                                                 
 gru (GRU)                   (None, 120)               46080     
                                                                 
 dense_0 (Dense)             (None, 50)                6050      
                                                                 
 dense_1 (Dense)             (None, 10)                510       
                                                                 
 output_softmax (Dense)      (None, 3)                 33        
                                                                 
Total params: 52673 (205.75 KB)
Trainable params: 52673 (205.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [2]:
import numpy as np

# bin = np.binary_repr(num, width=width)

# converts decimal number as shows up in modelsim to actual decimal number
# since verilog implementation uses fixed point and python does not
def fixed_to_dec(num, nfrac):
    return (num / (2 ** nfrac))

# vice versa
def dec_to_fixed(num, nfrac):
    return (num * 2 ** nfrac)

print(np.binary_repr(-1, width=16))
print(fixed_to_dec(-1, 10))
print(fixed_to_dec(np.array([-1, -1]), 10))

1111111111111111
-0.0009765625
[-0.00097656 -0.00097656]


In [3]:
print(dec_to_fixed(-0.56890726, 10))
print(dec_to_fixed(0.00138, 10))
print(fixed_to_dec(903, 10))
print(fixed_to_dec(9613, 10))

-582.56103424
1.41312
0.8818359375
9.3876953125


In [4]:
import tensorflow as tf

# grab GRU layer from overall model
gru = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("gru").output
)

# constant -0.0009765625 input
x = np.full((1, 15, 6), -0.0009765625, dtype=np.float32)

gru_output = gru.predict(x)

print("GRU output shape:", gru_output.shape)
print("GRU output for -1 input:", gru_output)

# constant 0 input
x = np.full((1, 15, 6), 0, dtype=np.float32)

gru_output = gru.predict(x)

print("GRU output shape:", gru_output.shape)
print("GRU output for 0 input:", gru_output)

1/1 [==============================] - 0s 337ms/step
GRU output shape: (1, 120)
GRU output for -1 input: [[-0.56890726 -0.09567672 -0.34913725 -0.99494064  0.3613983   0.19687709
  -0.05769272 -0.14897995  0.1877181  -0.03550649 -0.24532282  0.10750702
  -0.11598816 -0.18015094 -0.16135089 -0.85767335 -0.27906138  0.7703383
  -0.5340335   0.05125267 -0.25609422 -0.25098908  0.4315106  -0.37832123
  -0.20760763  0.7366304  -0.07465351  0.7162866   0.43648556  0.05374935
  -0.70293635  0.7977078   0.28382617  0.27658048 -0.54644835  0.12203787
   0.01980045  0.13664083  0.20438394  0.8159071  -0.27387103 -0.2850782
  -0.6261986  -0.11498982 -0.68637383  0.04770102 -0.1362696   0.09853638
  -0.81951106  0.65336573 -0.50525874 -0.1107319  -0.17950605  0.00516415
  -0.18323974 -0.647145   -0.05634918  0.03898488  0.03517921 -0.11741739
  -0.5549089   0.50507605 -0.02983442 -0.88004863 -0.06596029  0.03512058
   0.6822395  -0.19708467 -0.32967156 -0.45299914  0.52376145  0.25188696
  -0.6143

In [5]:
import numpy as np

# =======================
# set config
# =======================
TIMESTEPS = 15
INPUT_SIZE = 6
HIDDEN_SIZE = 120

# =======================
# helper functions
# =======================
def load_vector(filename):
    return np.loadtxt(filename)

def load_matrix(filename, rows, cols):
    data = np.loadtxt(filename)
    return data.reshape(rows, cols)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# =======================
# get GRU weights
# =======================

gru = model.layers[1]

kernel, recurrent_kernel, bias = gru.get_weights()

# split kernels
W_z, W_r, W_h = np.split(kernel, 3, axis=1)
U_z, U_r, U_h = np.split(recurrent_kernel, 3, axis=1)

# split bias into input and recurrent
b_input = bias[0]   # (360,)
b_rec   = bias[1]   # (360,)

# split per gate (update, reset, candidate hidden state)
b_z, b_r, b_h = np.split(b_input, 3)
b_z_rec, b_r_rec, b_h_rec = np.split(b_rec, 3)

# =======================
# set input
# =======================
x = np.zeros((TIMESTEPS, INPUT_SIZE)) # all zeroes

# =======================
# initialize hidden state
# =======================
h = np.zeros(HIDDEN_SIZE) # more zeroes

# trace storage
z_trace = []
z_raw_trace = []
r_trace = []
r_raw_trace = []
h_tilde_trace = []
h_tilde_raw_trace = []
h_trace = []

# =======================
# GRU calculation
# =======================
for t in range(TIMESTEPS):
    x_t = x[t]

    # gates
    z_raw = x_t @ W_z + h @ U_z + b_z + b_z_rec
    r_raw = x_t @ W_r + h @ U_r + b_r + b_r_rec
    z = sigmoid(z_raw)
    r = sigmoid(r_raw)

    # candidate (reset_after=True)
    h_tilde_raw = x_t @ W_h + b_h + r * (h @ U_h + b_h_rec)
    h_tilde = np.tanh(h_tilde_raw)

    # hidden state update
    h = (1 - z) * h_tilde + z * h

    # save traces
    z_trace.append(z)
    z_raw_trace.append(z_raw)
    r_trace.append(r)
    r_raw_trace.append(r_raw)
    h_tilde_trace.append(h_tilde)
    h_tilde_raw_trace.append(h_tilde_raw)
    h_trace.append(h)

# convert to arrays
z_raw_trace = np.array(z_raw_trace)
z_trace = np.array(z_trace)
r_raw_trace = np.array(r_raw_trace)
r_trace = np.array(r_trace)
h_tilde_trace = np.array(h_tilde_trace)
h_tilde_raw_trace = np.array(h_tilde_raw_trace)
h_trace = np.array(h_trace)

# =======================
# outputs
# =======================
print("final hidden state (first 10 units):")
print(h[:10])

print("\nhidden state evolution (first unit):")
print(h_trace[:, 0])

final hidden state (first 10 units):
[-0.56835281 -0.09412968 -0.35088817 -0.99496819  0.3640165   0.19295665
 -0.05823543 -0.15016247  0.18848199 -0.0339588 ]

hidden state evolution (first unit):
[ 0.03352435 -0.01083271 -0.09597818 -0.20979171 -0.31225219 -0.38451959
 -0.43647438 -0.47690711 -0.50939999 -0.53521844 -0.55493816 -0.56869199
 -0.57617964 -0.57659717 -0.56835281]


In [6]:
print("final hidden state (first 10 units):")
print(h[:10])

print("final hidden state (first 10 units) in modelsim:")
print(dec_to_fixed(h[:10], 6))

print("\nhidden state evolution (first unit):")
print(h_trace[:, 0])

print("\nr_raw[0]:")
print(r_raw_trace[0, :10], 6)
print("r[0]:")
print(r_trace[0, :10], 6)

print("\nr_raw[0] in modelsim:")
print(dec_to_fixed(r_raw_trace[0, :10], 6))
print("\nz_raw[0] in modelsim:")
print(dec_to_fixed(z_raw_trace[0, :10], 6))

print("\nr[0] in modelsim:")
print(dec_to_fixed(r_trace[0, :10], 6))
print("\nz[0] in modelsim:")
print(dec_to_fixed(z_trace[0, :10], 6))

print("\nh_tilde_raw[0] in modelsim:")
print(dec_to_fixed(h_tilde_raw_trace[0, :10], 6))
print("\nh_tilde[0] in modelsim:")
print(dec_to_fixed(h_tilde_trace[0, :10], 6))

print("\nh_1[0] in modelsim:")
print(dec_to_fixed(h_trace[0, :10], 6))

final hidden state (first 10 units):
[-0.56835281 -0.09412968 -0.35088817 -0.99496819  0.3640165   0.19295665
 -0.05823543 -0.15016247  0.18848199 -0.0339588 ]
final hidden state (first 10 units) in modelsim:
[-36.37457998  -6.02429944 -22.45684266 -63.67796418  23.29705569
  12.34922579  -3.72706771  -9.61039822  12.06284759  -2.17336339]

hidden state evolution (first unit):
[ 0.03352435 -0.01083271 -0.09597818 -0.20979171 -0.31225219 -0.38451959
 -0.43647438 -0.47690711 -0.50939999 -0.53521844 -0.55493816 -0.56869199
 -0.57617964 -0.57659717 -0.56835281]

r_raw[0]:
[ 0.01607639  0.07912929 -0.02188805  0.16814783  0.11022233  0.09159613
  0.09734208  0.40633118  0.06882744  0.07031781] 6
r[0]:
[0.50401901 0.51977201 0.49452821 0.54193819 0.52752772 0.52288304
 0.52431632 0.60020784 0.51720007 0.51757221] 6

r_raw[0] in modelsim:
[ 1.02888894  5.06427431 -1.40083539 10.76146126  7.05422926  5.8621521
  6.22989321 26.00519562  4.40495586  4.50033998]

z_raw[0] in modelsim:
[-45.731021

In [7]:
print(fixed_to_dec(np.array([-35, -5, -41, -57, 23, 2, -18, -22, -37, -15]), 6))
print(h[:10])

print(np.size([-35, -5, -41, -57, 23, 2, -18, -22, -37, -15, -16, 1, -19,
                                  -18, -32, -51, -4, 1, -44, -12, -39, -47, -35, -8, 0, 5,
                                  -50, 12, 0, -17, -23, -4, -13, -1, -56, 53, -7, -29, -28,
                                  -1, -23, -32, -61]))

hls = [-0.8125  ,  0.015625, -0.6875  , -1.      ,  0.015625, -0.328125,
        -0.25    , -0.09375 , -0.46875 , -0.296875, -0.265625, -0.109375,
        -0.125   , -0.265625, -0.328125, -0.734375, -0.390625,  0.1875  ,
        -0.453125, -0.265625, -0.53125 , -0.078125, -0.34375 , -0.421875,
        -0.1875  ,  0.28125 , -0.53125 ,  0.125   ,  0.140625, -0.078125,
        -0.8125  ,  0.3125  ,  0.15625 , -0.015625, -0.65625 ,  0.40625 ,
        -0.03125 , -0.203125, -0.171875,  0.4375  , -0.953125, -0.71875 ,
        -0.78125 ,  0.015625, -0.78125 , -0.09375 , -0.21875 , -0.640625,
        -0.953125,  0.859375, -0.515625, -0.234375, -0.453125, -0.296875,
        -0.296875, -0.953125, -0.109375, -0.328125, -0.796875, -0.875   ,
        -0.53125 ,  0.109375, -0.140625, -0.953125, -0.046875, -0.109375,
         0.171875, -0.4375  , -0.3125  , -0.765625, -0.125   , -0.890625,
        -0.71875 , -0.1875  ,  0.03125 , -0.140625, -0.65625 , -0.484375,
        -0.109375, -0.03125 , -0.21875 , -0.125   , -0.703125, -0.78125 ,
         0.453125, -0.671875, -0.8125  , -0.140625, -0.5625  , -0.0625  ,
        -0.15625 ,  0.34375 , -0.796875, -0.59375 , -0.765625, -0.59375 ,
        -0.390625, -0.703125, -0.96875 ,  0.34375 , -0.53125 , -0.6875  ,
        -0.359375,  0.15625 , -0.359375, -0.921875, -0.375   , -0.671875,
        -0.4375  , -0.078125, -0.171875, -0.375   , -0.0625  ,  0.015625,
         0.15625 ,  0.03125 ,  0.578125, -0.328125, -0.734375,  0.015625]

handmade = fixed_to_dec(np.array([-35, -5, -41, -57, 23, 2, -18, -22, -37, -15, -16, 1, -19,
                                  -18, -32, -51, -4, 1, -44, -12, -39, -47, -35, -8, 0, 5,
                                  -50, 12, 0, -17, -23, -4, -13, -1, -56, 53, -7, -29, -28,
                                  -1, -23, -32, -61]), 6)

print("difference from hls")
for i in range(43):
    print(np.abs(hls[i] - h[i]))

print("\ndifference from handmade")
for i in range (43):
    print(np.abs(handmade[i] - h[i]))

print("\nmax difference from hls")
print(np.max(np.abs(hls - h)))
print("\nmax difference from handmade")
print(np.max(np.abs(handmade - h[:43])))

print("\navg difference from hls")
print(np.mean(np.abs(hls - h)))
print("\navg difference from handmade")
print(np.mean(np.abs(handmade - h[:43])))

print("\nmax difference between hls and handmade")
print(np.max(np.abs(handmade - hls[:43])))

print("\navg difference between hls and handmade")
print(np.mean(np.abs(handmade - hls[:43])))

[-0.546875 -0.078125 -0.640625 -0.890625  0.359375  0.03125  -0.28125
 -0.34375  -0.578125 -0.234375]
[-0.56835281 -0.09412968 -0.35088817 -0.99496819  0.3640165   0.19295665
 -0.05823543 -0.15016247  0.18848199 -0.0339588 ]
43
difference from hls
0.24414718786447465
0.10975467872213908
0.33661183337540046
0.00503180962538563
0.34839149510091144
0.5210816528931064
0.1917645670985815
0.05641247222501514
0.6572319935846114
0.26291619701491475
0.018666717611018846
0.2195705118928414
0.010397097472996705
0.08627622030047127
0.16949707644660378
0.12384550538221195
0.11034475393500409
0.5829572314523764
0.07386215963687515
0.3212502213695018
0.2742467992580948
0.17493762528988777
0.777692897554644
0.04551209053826616
0.021046811196891835
0.4562671146925765
0.46337372447664943
0.5905067539359782
0.29537082663335135
0.13198804364002745
0.10969112647200785
0.48347880315301195
0.13144348572377523
0.29185532341410775
0.10768099825206279
0.2836771662174542
0.05148979190874042
0.3404021757504977
0.

In [8]:
print(handmade[3])
print(hls[3])
print(h[3])
print(gru_output[0][3])

-0.890625
-1.0
-0.9949681903746144
-0.9949682


In [9]:
print("r evolution in modelsim:")
print(dec_to_fixed(r_trace[:, :10], 6))

print("\nz evolution in modelsim:")
print(dec_to_fixed(z_trace[:, :10], 6))

print("\nh_tilde evolution in modelsim:")
print(dec_to_fixed(h_tilde_trace[:, :10], 6))

print("\nh evolution in modelsim:")
print(dec_to_fixed(h_trace[:, :10], 6))

r evolution in modelsim:
[[32.2572167  33.26540837 31.64980513 34.68404431 33.76177403 33.46451425
  33.55624465 38.41330173 33.10080444 33.12462163]
 [34.91749136 32.89578328 29.49721626 38.11676108 37.27973068 35.78348244
  35.87591859 48.96498263 35.79273242 35.45116869]
 [37.73560433 33.24800785 30.81225315 46.66037876 39.97440107 34.01987968
  39.50830287 53.74897279 38.84469363 35.04071599]
 [40.39374569 33.53261266 35.60436375 54.53369566 42.63922921 33.23948146
  43.38696984 55.37170267 42.43628359 33.67219473]
 [41.94860723 34.0717082  41.51339023 59.02927871 44.18116012 34.52994032
  46.59330013 54.51506782 45.7219857  31.95978704]
 [42.39556623 35.33287321 45.30872745 60.866111   43.88637039 36.8975067
  47.98634015 51.41236437 48.0523662  29.92042749]
 [41.71732134 37.15412885 46.47786712 61.5400704  42.29364311 39.99986536
  47.42252567 46.70354194 49.14227464 27.42399862]
 [40.26632264 39.21728925 45.72631412 61.72913168 40.19317023 43.49519628
  45.3084952  41.46653171 4

In [10]:
print("evolution in modelsim (by timestep):\n")

num_steps = min(15, r_trace.shape[1])

for t in range(num_steps):
    print(f"t = {t}")
    
    r_fix = dec_to_fixed(r_trace[t, :5], 6)
    z_fix = dec_to_fixed(z_trace[t, :5], 6)
    h_tilde_fix = dec_to_fixed(h_tilde_trace[t, :5], 6)
    h_fix = dec_to_fixed(h_trace[t, :5], 6)
    
    r_raw = dec_to_fixed(r_raw_trace[t, :5], 6)
    z_raw = dec_to_fixed(z_raw_trace[t, :5], 6)
    h_tilde_raw = dec_to_fixed(h_tilde_raw_trace[t, :5], 6)
    
    print(f"  r_raw   : {r_raw}")
    print(f"  r       : {r_fix}")
    
    print(f"  z_raw   : {z_raw}")
    print(f"  z       : {z_fix}")
    
    print(f"  h_tilde_raw : {h_tilde_raw}")
    print(f"  h_tilde : {h_tilde_fix}")
    
    print(f"  h       : {h_fix}")
    print()



evolution in modelsim (by timestep):

t = 0
  r_raw   : [ 1.02888894  5.06427431 -1.40083539 10.76146126  7.05422926]
  r       : [32.2572167  33.26540837 31.64980513 34.68404431 33.76177403]
  z_raw   : [-45.73102188   4.78328991  20.84177589  -0.66530693   8.910676  ]
  z       : [21.03007047 33.19526614 37.16487994 31.83367477 34.22407739]
  h_tilde_raw : [ 3.19828395 -0.58049076 -0.93051209  6.4176036   6.56592707]
  h_tilde : [ 3.19562423 -0.58047484 -0.93044652  6.39617991  6.54298765]
  h       : [ 2.14555856 -0.27939645 -0.39013507  3.21471255  3.04411709]

t = 1
  r_raw   : [ 11.70246228   3.58406948 -10.03162332  24.77174372  21.31374959]
  r       : [34.91749136 32.89578328 29.49721626 38.11676108 37.27973068]
  z_raw   : [-40.53246215  -9.08658202  26.17417869  -8.15781993  35.0122086 ]
  z       : [22.19252369 29.73216275 38.45383989 29.96330189 40.54109166]
  h_tilde_raw : [-2.20109903 -1.1447928   1.20108489  1.18144143 11.41896028]
  h_tilde : [-2.20023161 -1.14467072  

In [11]:
t = 14

print(f"t = {t}")
    
r_fix = dec_to_fixed(r_trace[t, :15], 6)
z_fix = dec_to_fixed(z_trace[t, :15], 6)
h_tilde_fix = dec_to_fixed(h_tilde_trace[t, :15], 6)
h_fix = dec_to_fixed(h_trace[t, :15], 6)
    
r_raw = dec_to_fixed(r_raw_trace[t, :15], 6)
z_raw = dec_to_fixed(z_raw_trace[t, :15], 6)
h_tilde_raw = dec_to_fixed(h_tilde_raw_trace[t, :15], 6)
    
print(f"  r_raw   : {r_raw}")
print(f"  r       : {r_fix}")
    
print(f"  z_raw   : {z_raw}")
print(f"  z       : {z_fix}")
    
print(f"  h_tilde_raw : {h_tilde_raw}")
print(f"  h_tilde : {h_tilde_fix}")
    
print(f"  h       : {h_fix}")
print()

t = 14
  r_raw   : [ 18.45602777  61.55869122 -19.40894954  94.94041804 -29.82661752
 124.58627994  -6.60238539 -47.72988274  39.03327005 -57.93391225
 -56.19790058 -49.60253061 -93.04572601 -45.14297672 -38.09497406]
  r       : [36.58229541 46.30354965 27.18461193 52.16590194 24.67543916 56.00525185
 30.35086596 20.59146491 41.46667612 18.43068836 18.78868983 20.18495757
 12.12232776 21.16000889 22.74782988]
  z_raw   : [  88.18461642  -22.99171673  142.06366644   58.99449779   50.75926606
    6.33222683  156.52967164   27.53966063  279.90861015 -114.42693627
   51.60709804  107.91982423   38.68065954   87.08664892  109.06512363]
  z       : [51.11366707 26.31310102 57.72855297 45.7860155  44.0639664  33.58176655
 58.89612911 38.7806094  63.20325156  9.17298041 44.24534361 53.99871996
 41.38617299 50.93620069 54.14880016]
  h_tilde_raw : [ -38.27439342   -1.93623588  -33.17206603 -165.80471315   12.27599875
   14.93913781    1.71317992   -8.79495487    8.34016024   -2.51117813
  -24.